In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [27]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
import os
import pandas as pd
import json
import re

# ========================
# 1. Model & Tokenizer Initialization
# ========================
tokenizer = AutoTokenizer.from_pretrained("impyadav/GPT2-FineTuned-Hinglish-Song-Generation")
model = AutoModelForCausalLM.from_pretrained("impyadav/GPT2-FineTuned-Hinglish-Song-Generation")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token

special_tokens = [
    "[Sentiment]", "[Rhyme]", "[Tempo]", "[Energy]", "[Loudness]",
    "[Danceability]", "[Speechiness]", "[Acousticness]", "[Instrumentalness]",
    "[Liveness]", "[Valence]", "[RhymeScheme]", "[Stanza]", 
    "[EndStanza]", "[Line]", "[EndLine]", "[A]", "[B]", "[C]", "[D]", "[E]", 
    "[F]", "[G]", "[H]", "[I]", "[J]", "[K]", "[L]", "[M]", "[N]", "[O]", "[P]"
]
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})
model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ========================
# 2. Prepare the Dataset
# ========================

file_path = "/kaggle/input/final-hindi-dataset/hindi_songs_with_sentiment.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
df = pd.DataFrame(data)


df = df.iloc[1200:1600]

class HinglishLyricsDataset(Dataset):
    def __init__(self, df, tokenizer, max_seq_len=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)
    
    def parse_lyrics_with_rhyme(self, lyrics_str, rhyme_scheme, rhyme_dict):
        # Check if lyrics is a string representation of a list
        if isinstance(lyrics_str, str) and lyrics_str.startswith('[') and lyrics_str.endswith(']'):
            try:
                # Remove the 'Lyrics' entry and any numeric entry at the end if present
                lyrics_list = eval(lyrics_str)
                if lyrics_list and lyrics_list[0] == 'Lyrics':
                    lyrics_list = lyrics_list[1:]
                # Remove any numeric entries at the end
                while lyrics_list and lyrics_list[-1].isdigit():
                    lyrics_list.pop()
            except:
                # If eval fails, split by commas and clean up
                lyrics_list = lyrics_str.strip('[]').split(',')
                lyrics_list = [line.strip().strip("'\"") for line in lyrics_list]
        else:
            # If it's already a clean string, just split by newlines
            lyrics_list = lyrics_str.split('\n')
        
        # Clean up empty lines and prepare for stanza detection
        cleaned_lyrics = []
        for line in lyrics_list:
            line = line.strip().strip("'\"")
            if line:  # Skip empty lines for structured format
                cleaned_lyrics.append(line)
        
        # Parse rhyme scheme to match with lines
        rhyme_labels = []
        if isinstance(rhyme_scheme, str) and rhyme_scheme.startswith('[') and rhyme_scheme.endswith(']'):
            try:
                rhyme_labels = eval(rhyme_scheme)
            except:
                rhyme_labels = rhyme_scheme.strip('[]').split(',')
                rhyme_labels = [label.strip().strip("'\"") for label in rhyme_labels]
        
        # Ensure we have the same number of rhyme labels as lyrics lines
        if len(rhyme_labels) != len(cleaned_lyrics):
            # Pad or truncate rhyme labels to match lyrics length
            if len(rhyme_labels) < len(cleaned_lyrics):
                rhyme_labels.extend(['X'] * (len(cleaned_lyrics) - len(rhyme_labels)))
            else:
                rhyme_labels = rhyme_labels[:len(cleaned_lyrics)]
        
        # Detect stanzas by looking for consecutive empty lines in original list
        stanza_breaks = []
        stanza_count = 0
        in_stanza = False
        
        for i, line in enumerate(lyrics_list):
            line = line.strip().strip("'\"")
            if line and not in_stanza:
                in_stanza = True
                stanza_count += 1
            elif not line and in_stanza:
                in_stanza = False
                stanza_breaks.append(i)
        
        # Format lyrics with stanza and rhyme information
        formatted_lyrics = "[Stanza]1[EndStanza]\n"
        current_stanza = 1
        stanza_line_count = 0
        
        for i, (line, rhyme) in enumerate(zip(cleaned_lyrics, rhyme_labels)):
            # Check if we need to start a new stanza
            if stanza_breaks and i >= stanza_breaks[0] and stanza_line_count > 0:
                stanza_breaks.pop(0)
                current_stanza += 1
                formatted_lyrics += f"\n[Stanza]{current_stanza}[EndStanza]\n"
                stanza_line_count = 0
            
            # Add the line with rhyme tag
            formatted_lyrics += f"[Line]{line}[{rhyme}][EndLine]\n"
            stanza_line_count += 1
        
        return formatted_lyrics, rhyme_dict

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        lyrics = row.get("Hindi Lyrics", "")
        sentiment = row.get("Sentiment", "NA")
        rhyme_dict = row.get("rhyme_dict", "{}")
        rhyme_scheme = row.get("rhyme_scheme", "[]")
        tempo = str(row.get("tempo", "NA"))
        energy = str(row.get("energy", "NA"))
        loudness = str(row.get("loudness", "NA"))
        danceability = str(row.get("danceability", "NA"))
        speechiness = str(row.get("speechiness", "NA"))
        acousticness = str(row.get("acousticness", "NA"))
        instrumentalness = str(row.get("instrumentalness", "NA"))
        liveness = str(row.get("liveness", "NA"))
        valence = str(row.get("valence", "NA"))
        explicit = str(row.get("explicit", "NA"))
        popularity = str(row.get("popularity", "NA"))
        chroma = str(row.get("chroma", "NA"))
        spectral_contrast = str(row.get("spectral_contrast", "NA"))
        zero_crossings = str(row.get("zero_crossings", "NA"))
        
        # Parse and format lyrics with rhyme scheme
        formatted_lyrics, rhyme_dict = self.parse_lyrics_with_rhyme(lyrics, rhyme_scheme, rhyme_dict)
        
        prompt = (
            f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
            f"[Rhyme]: {rhyme_dict} {tokenizer.eos_token} "
            f"[Tempo]: {tempo} {tokenizer.eos_token} "
            f"[Energy]: {energy} {tokenizer.eos_token} "
            f"[Loudness]: {loudness} {tokenizer.eos_token} "
            f"[Danceability]: {danceability} {tokenizer.eos_token} "
            f"[Speechiness]: {speechiness} {tokenizer.eos_token} "
            f"[Acousticness]: {acousticness} {tokenizer.eos_token} "
            f"[Instrumentalness]: {instrumentalness} {tokenizer.eos_token} "
            f"[Liveness]: {liveness} {tokenizer.eos_token} "
            f"[Valence]: {valence} {tokenizer.eos_token} "
            # f"[Explicit]: {explicit} {tokenizer.eos_token} "
            # f"[Popularity]: {popularity} {tokenizer.eos_token} "
            # f"[Chroma]: {chroma} {tokenizer.eos_token} "
            # f"[SpectralContrast]: {spectral_contrast} {tokenizer.eos_token} "
            # f"[ZeroCrossings]: {zero_crossings} {tokenizer.eos_token} "
            # f"[RhymeScheme]: {rhyme_scheme}\n"
        )
        
        combined_text = prompt + formatted_lyrics
        combined_enc = self.tokenizer(
            combined_text,
            return_tensors='pt',
            padding="max_length",
            truncation=True,
            max_length=self.max_seq_len
        )
        
        input_ids = combined_enc.input_ids.squeeze()
        attention_mask = combined_enc.attention_mask.squeeze()

        prompt_len = len(self.tokenizer.encode(prompt, add_special_tokens=False))
        labels = input_ids.clone()
        labels[:prompt_len] = -100

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

max_seq_len = 512
batch_size = 4
dataset_obj = HinglishLyricsDataset(df, tokenizer, max_seq_len=max_seq_len)
dataloader = DataLoader(dataset_obj, batch_size=batch_size, shuffle=True)

# ========================
# 3. Training Setup
# ========================

optimizer = AdamW(model.parameters(), lr=5e-5)
scaler = GradScaler()
epochs = 4
total_steps = len(dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

accumulation_steps = 8

# ========================
# 4. Training Loop
# ========================

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for i, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss.mean() / accumulation_steps

        scaler.scale(loss).backward()
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        running_loss += loss.item() * accumulation_steps
        if (i + 1) % 50 == 0:
            avg_loss = running_loss / 50
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(dataloader)}], Loss: {avg_loss:.4f}")
            running_loss = 0.0
    
    # Save checkpoint at the end of each epoch
    save_checkpoint_path = f"./checkpoints/epoch_{epoch+1}"
    os.makedirs(save_checkpoint_path, exist_ok=True)
    model.save_pretrained(save_checkpoint_path)
    tokenizer.save_pretrained(save_checkpoint_path)
    print(f"Checkpoint saved at epoch {epoch+1}")
    
    torch.cuda.empty_cache()

print("Training complete.")

# ========================
# 5. Save the Fine-Tuned Model & Tokenizer
# ========================

save_path = "./fine_tuned_impyadav_GPT2_hindi_lyrics"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# ========================
# 6. Generate Rhyming Dictionary
# ========================

import re
from collections import OrderedDict

def generate_rhyme_dict(rhyme_pattern="AABB", custom_endings=None):
    """Return a mapping of pattern-letters to Hindi Latin-script endings."""
    # Base set of Hindi endings (latin transliteration)
    base = {
        'A': 'a', 'B': 'aa', 'C': 'i', 'D': 'ee', 'E': 'u', 'F': 'oo',
        'G': 'e', 'H': 'ai', 'I': 'o', 'J': 'au', 'K': 'an', 'L': 'aan',
        'M': 'in', 'N': 'een', 'O': 'on', 'P': 'oon', 'Q': 'ar', 'R': 'aar',
        'S': 'ur', 'T': 'oor', 'U': 'eer', 'V': 'esh', 'W': 'ish',
        'X': 'ya', 'Y': 'iya', 'Z': 'ana'
    }
    # Merge custom if provided
    if custom_endings:
        base.update(custom_endings)

    # Preserve order of first appearance in pattern
    seen = list(OrderedDict.fromkeys(rhyme_pattern))
    return {lt: base.get(lt, '') for lt in seen}


def detect_rhyme_scheme(lines, rhyme_dict, anchor='any'):
    """
    Given a list of text lines and a rhyme_dict mapping letters->endings,
    return a rhyme scheme string by finding matching endings in each line.

    anchor: 'end' to match only line-endings, 'any' to scan all words.
    """
    scheme = []
    # Precompile endings sorted by length desc to match longest first
    endings = sorted(rhyme_dict.values(), key=len, reverse=True)

    for line in lines:
        word_ends = []
        # find endings anywhere
        for ending in endings:
            pattern = re.compile(rf"(\b\w*{re.escape(ending)}\b)", re.IGNORECASE)
            matches = pattern.findall(line)
            if matches:
                # store (ending, position) for selection
                for m in matches:
                    pos = line.lower().find(m.lower())
                    word_ends.append((ending, pos))
        chosen = None
        if anchor == 'end':
            # match only if line ends with an ending
            for e in endings:
                if line.lower().rstrip(' .,;!?').endswith(e):
                    chosen = e
                    break
        else:
            # pick the match that occurs latest (highest pos), or longest
            if word_ends:
                word_ends.sort(key=lambda x: (x[1], len(x[0])))
                chosen = word_ends[-1][0]

        # map chosen ending back to its letter
        if chosen:
            # find first letter with this ending
            letter = next((lt for lt, end in rhyme_dict.items() if end == chosen), '?')
        else:
            letter = '?'
        scheme.append(letter)

    return ''.join(scheme)

# ========================
# 7. Enhanced Generation Function
# ========================

def generate_lyrics(
    seed, 
    sentiment, 
    rhyme_pattern="AABB", 
    tempo=120, 
    energy=0.5,
    loudness=-30,
    danceability=0.5,
    num_stanzas=2,
    lines_per_stanza=4,
    max_len=512
):
    model.eval()
    
    # Generate rhyme dictionary based on the pattern
    rhyme_dict = generate_rhyme_dict(rhyme_pattern)
    
    prompt = (
        f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
        f"[Rhyme]: {rhyme_dict} {tokenizer.eos_token} "
        f"[Tempo]: {tempo} {tokenizer.eos_token} "
        f"[Energy]: {energy} {tokenizer.eos_token} "
        f"[Loudness]: {loudness} {tokenizer.eos_token} "
        f"[Danceability]: {danceability} {tokenizer.eos_token} "
        f"[RhymeScheme]: {rhyme_pattern}\n"
        f"[Stanza]1[EndStanza]\n"
        f"{seed}"
    )
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    
    # Define stopping criteria to avoid generating beyond a reasonable length
    stop_token_ids = [tokenizer.eos_token_id]
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_len,
            num_return_sequences=1,
            temperature=1.0,
            top_p=0.95,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3
        )
    
    raw_lyrics = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Post-process to format the lyrics nicely
    formatted_lyrics = post_process_lyrics(raw_lyrics)
    
    return formatted_lyrics

def post_process_lyrics(raw_lyrics):
    """
    Post-process the raw generated lyrics to format them nicely with stanzas and rhyme schemes
    """
    # Extract only the generated lyrics part (after the prompt)
    lines = raw_lyrics.split('\n')
    start_idx = 0
    for i, line in enumerate(lines):
        if "[Stanza]1[EndStanza]" in line:
            start_idx = i
            break
    
    # Extract lyrics content
    lyrics_content = lines[start_idx:]
    
    # Clean up formatting
    formatted_output = ""
    current_stanza = 0
    
    for line in lyrics_content:
        # Handle stanza markers
        if "[Stanza]" in line:
            stanza_match = re.search(r'\[Stanza\](\d+)\[EndStanza\]', line)
            if stanza_match:
                stanza_num = stanza_match.group(1)
                if current_stanza > 0:  # Add extra newline between stanzas
                    formatted_output += "\n"
                formatted_output += f"Stanza_{stanza_num}:\n"
                current_stanza = int(stanza_num)
        
        # Handle line content with rhyme scheme
        elif "[Line]" in line:
            # Extract line content and rhyme tag
            line_match = re.search(r'\[Line\](.*?)\[(.*?)\]\[EndLine\]', line)
            if line_match:
                line_content = line_match.group(1)
                rhyme_tag = line_match.group(2)
                formatted_output += f"{line_content} ({rhyme_tag})\n"
        
        # Handle any other line that might have been generated without proper tags
        elif line.strip() and not any(tag in line for tag in ["[Sentiment]", "[Rhyme]", "[Tempo]"]):
            formatted_output += f"{line}\n"
    
    return formatted_output

# ========================
# 8. Example Generation
# ========================

# Example 1: Romantic song with AABB rhyme pattern
romantic_lyrics = generate_lyrics(
    seed="Tere bina", 
    sentiment="Romantic, love, passion", 
    rhyme_pattern="AABB", 
    tempo=90,
    energy=0.4,
    danceability=0.6
)
print("Generated Romantic Lyrics:\n", romantic_lyrics)

# Example 2: Sad song with ABAB rhyme pattern
sad_lyrics = generate_lyrics(
    seed="Dil ke", 
    sentiment="Sad, melancholy, heartbreak", 
    rhyme_pattern="ABAB", 
    tempo=70,
    energy=0.3,
    danceability=0.4
)

print("\nGenerated Sad Lyrics:\n", sad_lyrics)

# Example 3: Party song with AAAA rhyme pattern
party_lyrics = generate_lyrics(
    seed="Aaj raat", 
    sentiment="Happy, energetic, celebration", 
    rhyme_pattern="AAAA", 
    tempo=130,
    energy=0.8,
    danceability=0.9
)
print("\nGenerated Party Lyrics:\n", party_lyrics)

/tmp/ipykernel_31/1734216792.py:202: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_31/1734216792.py:223: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1/4], Step [50/100], Loss: 6.5284
Epoch [1/4], Step [100/100], Loss: 5.1167
Checkpoint saved at epoch 1


/tmp/ipykernel_31/1734216792.py:223: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [2/4], Step [50/100], Loss: 4.4939
Epoch [2/4], Step [100/100], Loss: 4.2997
Checkpoint saved at epoch 2


/tmp/ipykernel_31/1734216792.py:223: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [3/4], Step [50/100], Loss: 4.1014
Epoch [3/4], Step [100/100], Loss: 3.9841
Checkpoint saved at epoch 3


/tmp/ipykernel_31/1734216792.py:223: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [4/4], Step [50/100], Loss: 3.8590
Epoch [4/4], Step [100/100], Loss: 3.7483
Checkpoint saved at epoch 4
Training complete.
Generated Romantic Lyrics:
 Stanza_1:
Tere bina apna sajano mein toh meri nazaron ne phir kam hai Ye hua yaad aaye tera majeure dil ko dolonne pichhe dekho Aabbar ka milti duniya mere pyar jaise goriya Toh tumhare naache mujha gayee taro maaf khudaanenge Kaun ho ooon rehana leke manzil jahaane ke wadaanenge
Palko pe fida aur yahi ki rani paai lageyate raahonge Kajal ye humdum din chupki shayari bekhudiye
Dard par banake tu diwaali udhale se jo palak thodaala dujaayenge
Laga laalon hi baaton main si sochta tha Main gaazi taonga chaandni vaadiyan thesasurat-chaanteen thesatisurat-laanteengaonka
Raat sachche bas hamaraaoongi sanmanuun samjeene waanto Kaisa lagati zahar detei daaho Toone bole judhi sur ek laddaare cheez badhat liyon uspe hone deshtaaya hai<|endoftext|>


Generated Sad Lyrics:
 Stanza_1:
Dil ke liye baaten se yeh haalat badee lagi  Achchha chhodk

In [42]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

# Model and Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/fine_tuned_impyadav_GPT2_hindi_lyrics")
model = AutoModelForCausalLM.from_pretrained("/kaggle/working/fine_tuned_impyadav_GPT2_hindi_lyrics")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Add special tokens if they don't already exist
special_tokens = [
    "[Sentiment]", "[Rhyme]", "[Tempo]", "[Energy]", "[Loudness]",
    "[Danceability]", "[Speechiness]", "[Acousticness]", "[Instrumentalness]",
    "[Liveness]", "[Valence]", "[Explicit]", "[Popularity]", "[Chroma]",
    "[SpectralContrast]", "[ZeroCrossings]", "[RhymeScheme]", "[Stanza]",
    "[EndStanza]", "[Line]", "[EndLine]"
]
tokens_to_add = [token for token in special_tokens if token not in tokenizer.get_vocab()]
if tokens_to_add:
    tokenizer.add_special_tokens({'additional_special_tokens': tokens_to_add})
    model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Common Hindi rhyme endings
common_endings = {
    'A': 'na', 'B': 'ye', 'C': 'ta', 'D': 'aa', 'E': 'ra', 'F': 'di', 'G': 'ki',
    'H': 'se', 'I': 'le', 'J': 'ho', 'K': 'ka', 'L': 'ja', 'M': 'ni', 'N': 'me',
    'O': 'ga', 'P': 'sa'
}

def format_lyrics(generated_text, words_per_line=5, lines_per_paragraph=4):
    """
    Format the generated lyrics by removing tags, splitting into lines with a specified number of words,
    and inserting paragraph breaks after a specified number of lines.

    Args:
        generated_text (str): The raw generated text from the model.
        words_per_line (int): Number of words per line (default: 5).
        lines_per_paragraph (int): Number of lines per paragraph (default: 4).

    Returns:
        str: Formatted lyrics.
    """
    # Remove the prompt section
    prompt_end = generated_text.find("[Stanza]1[EndStanza]")
    if prompt_end != -1:
        lyrics_text = generated_text[prompt_end + len("[Stanza]1[EndStanza]"):]
    else:
        lyrics_text = generated_text

    # Remove tags like [A], [B], [C], etc.
    lyrics_text = re.sub(r'\[[A-Z]\]', '', lyrics_text)
    lyrics_text = re.sub(r'\[EndLine\]', '', lyrics_text)
    lyrics_text = re.sub(r'\[\w+\]', '', lyrics_text)  # Remove any remaining tags like [G]

    # Split the text into words
    words = lyrics_text.split()

    # Group words into lines
    lines = []
    for i in range(0, len(words), words_per_line):
        line = ' '.join(words[i:i + words_per_line])
        lines.append(line)

    # Group lines into paragraphs
    paragraphs = []
    for i in range(0, len(lines), lines_per_paragraph):
        paragraph = '\n'.join(lines[i:i + lines_per_paragraph])
        paragraphs.append(paragraph)

    # Join paragraphs with double newlines
    formatted_lyrics = '\n\n'.join(paragraphs)

    return formatted_lyrics

def generate_lyrics(
    seed_phrase="",
    sentiment="Neutral",
    rhyme_pattern="AABB",
    tempo=120,
    energy=0.5,
    loudness=-30,
    danceability=0.5,
    speechiness=0.5,
    acousticness=0.5,
    instrumentalness=0.5,
    liveness=0.5,
    valence=0.5,
    explicit="False",
    popularity=50,
    chroma=0.5,
    spectral_contrast=0.5,
    zero_crossings=0.5,
    max_new_tokens=500,
    lines_per_stanza=15
):
    """
    Generate Hindi lyrics using a fine-tuned GPT-2 model.

    Args:
        seed_phrase (str): Starting phrase for the lyrics (optional).
        sentiment (str): Desired sentiment (e.g., "Romantic", "Sad").
        rhyme_pattern (str): Rhyme scheme (e.g., "AABB", "ABAB").
        tempo (float): Musical tempo (default: 120).
        energy (float): Energy level (0-1, default: 0.5).
        loudness (float): Loudness in dB (default: -30).
        danceability (float): Danceability score (0-1, default: 0.5).
        speechiness (float): Speechiness score (0-1, default: 0.5).
        acousticness (float): Acousticness score (0-1, default: 0.5).
        instrumentalness (float): Instrumentalness score (0-1, default: 0.5).
        liveness (float): Liveness score (0-1, default: 0.5).
        valence (float): Valence score (0-1, default: 0.5).
        explicit (str): Explicit content flag ("True" or "False", default: "False").
        popularityollis(int): Popularity score (0-100, default: 50).
        chroma (float): Chroma feature (0-1, default: 0.5).
        spectral_contrast (float): Spectral contrast (0-1, default: 0.5).
        zero_crossings (float): Zero crossings rate (0-1, default: 0.5).
        max_new_tokens (int): Number of new tokens to generate (default: 200).
        lines_per_stanza (int): Number of lines per stanza (default: 4).

    Returns:
        str: Formatted lyrics with stanzas and rhyme labels.
    """
    model.eval()

    # Generate rhyme_scheme_list and rhyme_dict
    rhyme_scheme_list = list(rhyme_pattern)  # e.g., ['A', 'A', 'B', 'B']
    unique_letters = sorted(set(rhyme_pattern))
    rhyme_dict = {letter: common_endings.get(letter, 'na') for letter in unique_letters}
    rhyme_dict_str = str(rhyme_dict)
    rhyme_scheme_str = str(rhyme_scheme_list)

    # Construct prompt
    prompt = (
        f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
        f"[Rhyme]: {rhyme_dict_str} {tokenizer.eos_token} "
        f"[Tempo]: {tempo} {tokenizer.eos_token} "
        f"[Energy]: {energy} {tokenizer.eos_token} "
        f"[Loudness]: {loudness} {tokenizer.eos_token} "
        f"[Danceability]: {danceability} {tokenizer.eos_token} "
        f"[Speechiness]: {speechiness} {tokenizer.eos_token} "
        f"[Acousticness]: {acousticness} {tokenizer.eos_token} "
        f"[Instrumentalness]: {instrumentalness} {tokenizer.eos_token} "
        f"[Liveness]: {liveness} {tokenizer.eos_token} "
        f"[Valence]: {valence} {tokenizer.eos_token} "
        # f"[Explicit]: {explicit} {tokenizer.eos_token} "
        # f"[Popularity]: {popularity} {tokenizer.eos_token} "
        # f"[Chroma]: {chroma} {tokenizer.eos_token} "
        # f"[SpectralContrast]: {spectral_contrast} {tokenizer.eos_token} "
        # f"[ZeroCrossings]: {zero_crossings} {tokenizer.eos_token} "
        f"[RhymeScheme]: {rhyme_scheme_str}\n"
        "[Stanza]1[EndStanza]\n"
    )
    if seed_phrase:
        prompt += seed_phrase

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)

    # Generate
    with torch.no_grad():
        generated_ids = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            do_sample=True,
            temperature=0.6,
            top_p=0.92,
            top_k=70,
            repetition_penalty=1.15,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_return_sequences=1,
            no_repeat_ngram_size=2
        )

    # Decode
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    # print("Generated Text:\n", generated_text)
    # Extract lines with regex
    # pattern = r'\[Line\](.*?)\[(.*?)\]\[EndLine\]'
    # matches = re.findall(pattern, generated_text, re.DOTALL)

    # # Format lyrics
    # formatted_lyrics = ""
    # stanza_count = 1
    # for i, (line, rhyme) in enumerate(matches):
    #     if i % lines_per_stanza == 0:
    #         if i > 0:
    #             formatted_lyrics += "\n"
    #         formatted_lyrics += f"Stanza_{stanza_count}:\n"
    #         stanza_count += 1
    #     formatted_lyrics += f"{line.strip()} ({rhyme})\n"

    # # Fallback if no lines are extracted
    # if not matches:
    #     lines = [line.strip() for line in generated_text.split('\n') if line.strip() and not line.startswith('[')]
    #     for i, line in enumerate(lines):
    #         if i % lines_per_stanza == 0:
    #             if i > 0:
    #                 formatted_lyrics += "\n"
    #             formatted_lyrics += f"Stanza_{stanza_count}:\n"
    #             stanza_count += 1
    #         rhyme_label = rhyme_scheme_list[i % len(rhyme_scheme_list)]
    #         formatted_lyrics += f"{line} ({rhyme_label})\n"

    # return formatted_lyrics.strip()
    print(generated_text)
    return generated_text

# Example usage
if __name__ == "__main__":
    lyrics = generate_lyrics(
        seed_phrase="Tere bina",
        sentiment="Positive",
        rhyme_pattern="ABAB",
        tempo=70,
        energy=0.5,
        danceability=0.8
    )
    

    # Format the lyrics
    formatted_lyrics = format_lyrics(lyrics, words_per_line=6, lines_per_paragraph=5)

    # Print the result
    print("Formatted Lyrics:\n")
    print(formatted_lyrics)
    # print("Generated Romantic Lyrics:\n", lyrics)
    


[Sentiment]: Positive <|endoftext|> [Rhyme]: {'A': 'na', 'B': 'ye'} <|endoftext|> [Tempo]: 70 <|endoftext|> [Energy]: 0.5 <|endoftext|> [Loudness]: -30 <|endoftext|> [Danceability]: 0.8 <|endoftext|> [Speechiness]: 0.5 <|endoftext|> [Acousticness]: 0.5 <|endoftext|> [Instrumentalness]: 0.5 <|endoftext|> [Liveness]: 0.5 <|endoftext|> [Valence]: 0.5 <|endoftext|> [RhymeScheme]: ['A', 'B', 'A', 'B']
[Stanza]1[EndStanza]
Tere bina kabhi toh aaya hai, dhadkan ne takara haath mein humko deewanaan haathon me Inkaar se tujhe joh kehta hoon inkele yeh saans dil ko wahan haatta meInkaaron pe teraa rona khuda jaaye mujhe tumse pyariyan ki nazar ka phir na chhodo haan
2<|endoftext|>
Formatted Lyrics:

Tere bina kabhi toh aaya hai,
dhadkan ne takara haath mein humko
deewanaan haathon me Inkaar se tujhe
joh kehta hoon inkele yeh saans
dil ko wahan haatta meInkaaron pe

teraa rona khuda jaaye mujhe tumse
pyariyan ki nazar ka phir na
chhodo haan 2<|endoftext|>


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

# Load model and tokenizer
model_path = "./best_model_checkpoint"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def post_process_lyrics(raw_lyrics):
    """Clean and format generated lyrics, removing special tokens and structuring stanzas."""
    # Remove special tokens and metadata
    lines = raw_lyrics.split('\n')
    cleaned_lines = []
    current_stanza = []
    
    for line in lines:
        # Remove special tokens like [Sentiment], [Tempo], etc.
        line = re.sub(r'\[.*?\]', '', line)
        line = line.strip()
        if line:
            current_stanza.append(line)
            # Assume 4 lines per stanza (based on training setup)
            if len(current_stanza) == 4:
                cleaned_lines.extend(current_stanza)
                cleaned_lines.append("")  # Add blank line between stanzas
                current_stanza = []
    
    # Add any remaining lines
    if current_stanza:
        cleaned_lines.extend(current_stanza)
    
    # Join lines and remove extra whitespace
    return '\n'.join(line for line in cleaned_lines if line.strip()).strip()

def generate_lyrics(
    seed="Pyar ka", 
    sentiment="Romantic", 
    tempo=120, 
    energy=0.5,
    rhyme_pattern="AABB",
    num_stanzas=2,
    lines_per_stanza=4,
    max_len=512,
    initial_temp=1.0,
    temp_decay=0.95,
    min_temp=0.7
):
    """Generate Hindi song lyrics from audio features and sentiment."""
    # Build prompt matching training format
    prompt = (
        f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
        f"[Tempo]: {tempo} {tokenizer.eos_token} "
        f"[Energy]: {energy} {tokenizer.eos_token} "
        f"[RhymeScheme]: {rhyme_pattern}\n"
        f"[Stanza]1[EndStanza]\n"
        f"[LinesPerStanza]: {lines_per_stanza}\n"
        f"[TotalStanzas]: {num_stanzas}\n"
        f"{seed}"
    )
    
    # Tokenize prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt', truncation=True, max_length=max_len//2).to(device)
    
    # Generate lyrics with temperature scheduling
    generated = input_ids
    current_temp = initial_temp
    
    with torch.no_grad():
        for _ in range(max_len - len(input_ids[0])):
            outputs = model(input_ids=generated)
            next_token_logits = outputs.logits[:, -1, :]
            
            # Apply temperature
            next_token_logits = next_token_logits / current_temp
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            
            # Stop if EOS token is generated
            if next_token.item() == tokenizer.eos_token_id:
                break
                
            # Check stanza completion
            generated_text = tokenizer.decode(generated[0], skip_special_tokens=False)
            if "[EndStanza]" in generated_text:
                stanzas = generated_text.count("[EndStanza]")
                if stanzas >= num_stanzas:
                    break
            
            generated = torch.cat([generated, next_token], dim=-1)
            
            # Decay temperature
            current_temp = max(min_temp, current_temp * temp_decay)
    
    # Decode and post-process lyrics
    raw_lyrics = tokenizer.decode(generated[0], skip_special_tokens=False)
    formatted_lyrics = post_process_lyrics(raw_lyrics)
    
    return formatted_lyrics

# Example usage
if __name__ == "__main__":
    # Input parameters
    seed_text = "Dil se dil tak"
    sentiment = "Romantic"
    tempo = 100
    energy = 0.6
    rhyme_pattern = "ABCD"
    num_stanzas = 3
    lines_per_stanza = 4
    
    # Generate lyrics
    lyrics = generate_lyrics(
        seed=seed_text,
        sentiment=sentiment,
        tempo=tempo,
        energy=energy,
        rhyme_pattern=rhyme_pattern,
        num_stanzas=num_stanzas,
        lines_per_stanza=lines_per_stanza,
        initial_temp=1.2,
        temp_decay=0.9,
        min_temp=0.7
    )
    
    print("Generated Hindi Lyrics:")
    print(lyrics)

Generated Hindi Lyrics:
: Romantic <|endoftext|> : 100 <|endoftext|> : 0.6 <|endoftext|> : ABCD
1
: 4
: 3
Dil se dil tak hai kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya kya